# 🔴 Violence Detection System
**YOLOv8 + MediaPipe Pose + Transformer Encoder**

Run cells **in order**. Each section matches a phase from the README.

```
Phase 0 — Setup & GPU check
Phase 1 — Upload project files
Phase 2 — Preprocess videos → per-video .npy files
Phase 3 — Train the Transformer
Phase 4 — Evaluate on test set
Phase 5 — Run inference on a video
```

---
## Phase 0 — Setup & GPU Check

In [ ]:
# Verify GPU is available (should show Tesla T4 on Colab free tier)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Install all dependencies
!pip install -q \
    torch>=2.1.0 torchvision>=0.16.0 torchaudio>=2.1.0 \
    ultralytics>=8.3.0 \
    mediapipe>=0.10.0 \
    opencv-python-headless>=4.8.0 \
    scikit-learn>=1.3.0 \
    tensorboard>=2.14.0 \
    tqdm matplotlib seaborn pandas PyYAML

print('✅ Dependencies installed')

---
## Phase 1 — Upload Project Files

Upload your project files (all `.py` files) as a zip.

**Option A — Upload from your computer:**

In [ ]:
from google.colab import files
import os, zipfile

print('Upload your project zip (containing all .py files):')
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/')
        print(f'✅ Extracted {fname}')
    else:
        print(f'  Uploaded: {fname}')

In [ ]:
# --------------------------------------------------------------------------
# Option B — Mount Google Drive (if your project is already there)
# --------------------------------------------------------------------------
# Uncomment and run this cell instead of Option A if preferred

# from google.colab import drive
# drive.mount('/content/drive')
#
# PROJECT_DIR = '/content/drive/MyDrive/violence_detection'   # ← adjust path
# import os
# os.chdir(PROJECT_DIR)
# print('Working directory:', os.getcwd())

In [ ]:
# Set working directory to where the .py files live
import os
os.chdir('/content')

# Sanity check — these files must all be present
required = ['preprocess.py','dataset.py','transformer_model.py',
            'train.py','evaluate.py','detect_violence.py']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print('❌ Missing files:', missing)
    print('   Make sure you uploaded/extracted the full project.')
else:
    print('✅ All project files found')

# Create output directories
# [REQ-1] processed/ now holds violence/ and non_violence/ subdirs
for d in ['processed/violence', 'processed/non_violence',
          'checkpoints', 'logs', 'outputs']:
    os.makedirs(d, exist_ok=True)
print('✅ Output directories ready')

---
## Phase 2 — Preprocess Videos

Converts raw videos → **one `.npy` file per video** saved under:
```
processed/
├── violence/        ← one .npy per violent clip
└── non_violence/    ← one .npy per non-violent clip
```

**Dataset folder must look like this:**
```
dataset/
├── violence/        ← violent video clips
│   ├── vid1.mp4
│   └── ...
└── non_violence/    ← non-violent video clips
    ├── vid1.mp4
    └── ...
```

**Pipeline per frame:**
```
Video Frame → YOLO Person Detection → Best Person BBox → Crop ROI → MediaPipe Pose → Features
```

In [ ]:
# Upload dataset videos — run this if your dataset is NOT on Drive
# (For large datasets, Google Drive is much more practical)

import os
from google.colab import files

os.makedirs('/content/dataset/violence', exist_ok=True)
os.makedirs('/content/dataset/non_violence', exist_ok=True)

print('Upload ALL violence videos:')
upl = files.upload()
for fname, data in upl.items():
    with open(f'/content/dataset/violence/{fname}', 'wb') as f:
        f.write(data)

print('\nUpload ALL non-violence videos:')
upl = files.upload()
for fname, data in upl.items():
    with open(f'/content/dataset/non_violence/{fname}', 'wb') as f:
        f.write(data)

print('✅ Videos uploaded')

In [ ]:
# ── Run Preprocessing ──────────────────────────────────────────────────────
# [REQ-1] Outputs one .npy per video under processed/violence/ and processed/non_violence/
# [REQ-3] YOLO detects persons first; only the best-bbox ROI is cropped
# [REQ-4] MediaPipe pose runs only on the cropped ROI

!python preprocess.py \
    --dataset_path ./dataset \
    --output_path  ./processed \
    --sequence_length 30 \
    --yolo_model yolov8n.pt \
    --device auto

In [ ]:
# ── Verify preprocessing output [REQ-1] ────────────────────────────────────
# One .npy file per video — no monolithic X.npy / y.npy
import json
import os
from pathlib import Path

processed = Path('./processed')

violence_files     = sorted((processed / 'violence').glob('*.npy'))
non_violence_files = sorted((processed / 'non_violence').glob('*.npy'))

print(f'Violence     : {len(violence_files)} .npy files')
print(f'Non-violence : {len(non_violence_files)} .npy files')
print(f'Total        : {len(violence_files) + len(non_violence_files)} videos')

# Show a sample
import numpy as np
if violence_files:
    sample = np.load(str(violence_files[0]))
    print(f'\nSample .npy shape: {sample.shape}  (seq_len × feature_dim)')
    print(f'Example files:')
    for f in (violence_files[:3] + non_violence_files[:3]):
        print(f'  {f}')

with open('processed/metadata.json') as f:
    meta = json.load(f)
print(f'\nProcessing time : {meta["processing_time_seconds"]:.1f}s')
print(f'Processed       : {meta["processed"]} videos')
print(f'Skipped         : {meta["skipped"]} videos')

---
## Phase 3 — Train the Transformer

In [ ]:
# ── Launch Training ────────────────────────────────────────────────────────
# [REQ-1] Loads per-video .npy files dynamically via dataset.py
# [REQ-2] Fixed LR=1e-4 — no scheduler used at all
# Recommended settings for Colab T4: batch_size=64, num_workers=2

!python train.py \
    --processed_path ./processed \
    --epochs 50 \
    --batch_size 64 \
    --learning_rate 1e-4 \
    --patience 10 \
    --num_workers 2 \
    --embedding_dim 256 \
    --num_heads 8 \
    --num_layers 4 \
    --dropout 0.3

In [ ]:
# ── TensorBoard (optional — visualise loss/accuracy curves) ───────────────
%load_ext tensorboard
%tensorboard --logdir ./logs

In [ ]:
# ── Resume training from a checkpoint (if interrupted) ────────────────────
# Uncomment and run if you need to continue a previous run

# !python train.py \
#     --processed_path ./processed \
#     --epochs 100 \
#     --batch_size 64 \
#     --resume ./checkpoints/best_model.pth

In [ ]:
# List saved checkpoints
import os
ckpts = sorted(os.listdir('checkpoints'))
print('Saved checkpoints:')
for c in ckpts:
    size_mb = os.path.getsize(f'checkpoints/{c}') / 1e6
    print(f'  {c}  ({size_mb:.1f} MB)')

---
## Phase 4 — Evaluate on Test Set

In [ ]:
# ── Run Evaluation ─────────────────────────────────────────────────────────
!python evaluate.py \
    --checkpoint ./checkpoints/best_model.pth \
    --processed_path ./processed \
    --output_dir ./outputs \
    --threshold 0.5

In [ ]:
# ── Display evaluation plots inline ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plots = [
    ('outputs/confusion_matrix.png',       'Confusion Matrix'),
    ('outputs/roc_curve.png',              'ROC Curve'),
    ('outputs/precision_recall_curve.png', 'Precision-Recall Curve'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (path, title) in zip(axes, plots):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=13)
    else:
        ax.set_title(f'{title} — not found')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Print metrics summary ─────────────────────────────────────────────────
import json
with open('outputs/evaluation_results.json') as f:
    results = json.load(f)

m = results['metrics']
print('=== TEST SET RESULTS ===')
print(f"Accuracy  : {m['accuracy']:.4f}  ({m['accuracy']*100:.2f}%)")
print(f"Precision : {m['precision']:.4f}")
print(f"Recall    : {m['recall']:.4f}")
print(f"F1 Score  : {m['f1']:.4f}")
print(f"ROC-AUC   : {m['roc_auc']:.4f}")

---
## Phase 5 — Real-Time Inference on a Video

In [ ]:
# Upload a video to run inference on
from google.colab import files
print('Upload a video file to run violence detection on:')
uploaded = files.upload()
input_video = list(uploaded.keys())[0]
print(f'Using: {input_video}')

In [ ]:
# ── Run inference ──────────────────────────────────────────────────────────
# [REQ-3][REQ-4] Pose is extracted from YOLO-detected person ROI only

!python detect_violence.py \
    --video "{input_video}" \
    --checkpoint ./checkpoints/best_model.pth \
    --output ./outputs/annotated_output.mp4 \
    --threshold 0.5 \
    --yolo_model yolov8n.pt \
    --yolo_conf 0.4

In [ ]:
# ── Play the annotated output video inline ─────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import os

output_path = './outputs/annotated_output.mp4'

if os.path.exists(output_path):
    # Re-encode to H.264 so it plays in Colab
    !ffmpeg -y -i {output_path} -vcodec libx264 /tmp/output_h264.mp4 -loglevel quiet
    with open('/tmp/output_h264.mp4', 'rb') as f:
        video_data = b64encode(f.read()).decode()
    HTML(f'''
    <video width="800" controls>
        <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>
    ''')
else:
    print('Output video not found — check the inference step above for errors.')

In [ ]:
# ── Download the annotated video ───────────────────────────────────────────
from google.colab import files
files.download('./outputs/annotated_output.mp4')

---
## Bonus — Save Best Model to Google Drive

In [ ]:
# Copy best model checkpoint to Drive so it survives session resets
# Uncomment after mounting Drive (see Phase 1 Option B)

# import shutil
# shutil.copy('./checkpoints/best_model.pth',
#             '/content/drive/MyDrive/violence_detection/best_model.pth')
# print('✅ Checkpoint saved to Drive')

---
## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Reduce `--batch_size 16` or use `--yolo_model yolov8n.pt` |
| `No sequences generated` | Check dataset folder names are exactly `violence/` and `non_violence/` |
| `MediaPipe not detecting poses` | Lower `--yolo_conf 0.3` or use `--yolo_model yolov8s.pt` |
| `Training loss not decreasing` | Try `--learning_rate 5e-5` or more epochs |
| `No .npy files found` | Run preprocess.py first; check `processed/violence/` exists |
| Video won't play in Colab | Re-encode with the `ffmpeg` cell above |